In [1]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        query = f"SELECT * FROM {table_name};"
        
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.94.14.115


C:\Users\win 11\AppData\Local\Temp\ipykernel_24424\1204036107.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 7990 rows from 'extraction'


In [5]:
import pandas as pd

# 1. Create the copy
df = df_original.copy(deep=True)

# 2. Logic to extract the specific part of the filename
def extract_sub_dept(row):
    # Updated target list to match your full names
    target_depts = ['power-system', 'track-network', 'signalling-and-communication']
    
    if str(row['dept_name']).lower() in target_depts:
        try:
            # Splits by underscore and grabs index 3 (the 4th element)
            return row['filename'].split('_')[3]
        except (IndexError, AttributeError):
            return "Unknown_Format"
    return "Standard"

# 3. Apply extraction
df['filename_key'] = df.apply(extract_sub_dept, axis=1)

# --- REPORT 1: Department -> Filename Key -> Interval ---
report_with_interval = df.groupby(['dept_name', 'filename_key', 'interval']).size().reset_index(name='count')

# --- REPORT 2: Department -> Filename Key Only ---
report_filename_only = df.groupby(['dept_name', 'filename_key']).size().reset_index(name='count')

# 4. Display Results
print("### REPORT 1: Nested by Interval ###")
print(report_with_interval)

print("\n" + "="*40 + "\n")

print("### REPORT 2: Nested by Filename Only ###")
print(report_filename_only)

### REPORT 1: Nested by Interval ###
        dept_name         filename_key   interval  count
0    Power-System                  BLS     Weekly     11
1    Power-System    StationInspection  Quarterly    171
2    Power-System    StationInspection     Weekly   1769
3    Power-System    StationInspection     Yearly     30
4    Power-System                 TPSS  Quarterly      2
..            ...                  ...        ...    ...
67  Track-Network                 TMV3    Monthly      8
68  Track-Network                 TMV3     Yearly      1
69  Track-Network       TrainWashPlant    Monthly     45
70  Track-Network  TyreChangingMachine    Monthly     43
71  Track-Network              Walkway    Monthly    128

[72 rows x 4 columns]


### REPORT 2: Nested by Filename Only ###
                       dept_name          filename_key  count
0                   Power-System                   BLS     11
1                   Power-System     StationInspection   1970
2                   Power-

In [6]:
# Define the output file name
output_file = "department_reports.txt"

with open(output_file, 'w') as f:
    f.write("=== REPORT 1: DEPARTMENT > FILENAME KEY > INTERVAL ===\n")
    f.write(report_with_interval.to_string(index=False))
    
    f.write("\n\n" + "="*60 + "\n\n")
    
    f.write("=== REPORT 2: DEPARTMENT > FILENAME KEY ONLY ===\n")
    f.write(report_filename_only.to_string(index=False))

print(f"Reports successfully exported to {output_file}")

Reports successfully exported to department_reports.txt
